In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer
import os

In [ ]:
!ls slurm/data

In [ ]:
CACHE_ROOT = "/cs/labs/oabend/tomer.shahaf/hf_cache_root"
#research_df_tmp_with_cosines_parquet_path = os.path.join(CACHE_ROOT, "df_sampled_100k_tmp_with_cosines.pqt")
research_df_tmp_with_cosines_parquet_path = "/cs/labs/oabend/tomer.shahaf/sharelm_research/slurm/data/ours_dataset_medium_conversations_with_cosines.pqt"

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook' 

In [ ]:
long_conversations_df = pl.read_parquet(research_df_tmp_with_cosines_parquet_path)
long_conversations_df.shape

In [ ]:
long_conversations_df.columns

In [ ]:
px.histogram(long_conversations_df["count_before_model_semantic_change"])

In [ ]:
long_conversations_df["user_prompts"][0][1]

In [ ]:
df_clean = (
    long_conversations_df.lazy()  
    .with_row_index(name="conv_id") 
    .explode("user_prompts")
    .with_columns(
        pl.col("user_prompts").cum_count().over("conv_id").alias("turn_index")
    )
    .filter(pl.col("turn_index") < 5)
    .group_by("turn_index")
    .agg(pl.col("user_prompts")) 
    .collect() 
)
df_clean = df_clean.sort("turn_index")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

tfidf = TfidfVectorizer(stop_words='english', max_features=15)

corpus = [
    " ".join(row) 
    for row in df_clean.sort("turn_index")["user_prompts"].to_list()
]

tfidf_matrix = tfidf.fit_transform(corpus)
feature_names = tfidf.get_feature_names_out()

df_results = pd.DataFrame(
    tfidf_matrix.toarray().T, 
    index=feature_names, 
    columns=[f"Turn {i}" for i in sorted(df_clean["turn_index"])]
)

df_results.head()

In [ ]:
KEYWORDS_NUMBER = 10_000

corpus = [
    " ".join(row) 
    for row in df_clean.sort("turn_index")["user_prompts"].to_list()
]

vec = CountVectorizer(stop_words='english', max_features=KEYWORDS_NUMBER)
X = vec.fit_transform(corpus)

df_counts = pd.DataFrame(
    X.toarray(), 
    columns=vec.get_feature_names_out(),
    index=[f"Turn {i}" for i in sorted(df_clean["turn_index"])]
)

# 4. Normalize to get Probabilities (Distribution within the Turn)
# We divide each row by the sum of that row.
# Result: The values in each row now sum to 1.0
df_probs = df_counts.div(df_counts.sum(axis=1), axis=0)

# 5. Transpose for easier reading (Rows = Words, Cols = Turns)
df_final = df_probs.T

In [ ]:
# 6. Comparison: Calculate the Shift
# Positive 'Diff' = Word becomes MORE common in Turn 4
# Negative 'Diff' = Word becomes LESS common in Turn 4 (likely an opener word)
df_final['Diff (T4 - T1)'] = df_final['Turn 4'] - df_final['Turn 1']

# --- Display Results ---

print("--- Words that DROP the most (Openers) ---")
print(df_final.sort_values('Diff (T4 - T1)').head(10))

print("\n--- Words that RISE the most (Follow-ups) ---")
print(df_final.sort_values('Diff (T4 - T1)', ascending=False).head(10))

In [ ]:
drop_df = df_final.sort_values('Diff (T4 - T1)').head(10).reset_index().rename(columns={'index': 'Word'})
rise_df = df_final.sort_values('Diff (T4 - T1)', ascending=False).head(10).reset_index().rename(columns={'index': 'Word'})

In [ ]:
# show drop_df

# 1. Setup: Get the list of words we are currently plotting
words_to_plot = drop_df['Word'].tolist()

# 2. Get the specific data slices
# A. Probabilities (You already have this)
probs_subset = df_final.loc[words_to_plot, ['Turn 1', 'Turn 2', 'Turn 3', 'Turn 4']]

# B. Absolute Counts (Retrieve from the df_counts we created earlier)
# We transpose (.T) so rows are Words and columns are Turns, matching probs_subset
counts_subset = df_counts[words_to_plot].T 

# C. Multiplication Factors (Current Prob / Turn 1 Prob)
# If a word drops from 0.10 to 0.05, Factor is 0.5x
factors_subset = probs_subset.div(probs_subset['Turn 1'], axis=0)

# 3. Melt all three individually
# We use reset_index() so 'Word' becomes a column
melt_probs = probs_subset.reset_index().melt(id_vars='index', var_name='Turn', value_name='Probability').rename(columns={'index': 'Word'})
melt_counts = counts_subset.reset_index().melt(id_vars='index', var_name='Turn', value_name='Count').rename(columns={'index': 'Word'})
melt_factor = factors_subset.reset_index().melt(id_vars='index', var_name='Turn', value_name='Factor').rename(columns={'index': 'Word'})

# 4. Merge them into one "Rich" DataFrame
df_rich = melt_probs.merge(melt_counts, on=['Word', 'Turn'])\
                    .merge(melt_factor, on=['Word', 'Turn'])

# Preview to ensure it looks right
print(df_rich.head())

In [ ]:
fig = px.line(
    df_rich, 
    x="Turn", 
    y="Probability", 
    color="Word", 
    markers=True,
    title="Evolution of User Language - drop words",
    template="plotly_white",
    height=500,
    # --- CUSTOM HOVER CONFIGURATION ---
    hover_data={
        "Turn": False,             # Hide (already on x-axis)
        "Word": False,             # Hide (already in legend)
        "Probability": ":.4f",     # Format: 4 decimal places
        "Count": True,             # Show raw count
        "Factor": ":.2f"           # Format: 2 decimal places
    }
)

# Optional: Rename the labels in the tooltip to be friendlier
fig.update_traces(
    hovertemplate="<br>".join([
        "<b>%{x}</b>",
        "Prob: %{y:.4f}",
        "Count: %{customdata[1]}",    # Accesses 'Count'
        "Growth: %{customdata[2]:.2f}x" # Accesses 'Factor'
    ])
)

fig.update_layout(yaxis_title="Probability (Frequency)", hovermode="x unified")
fig.show()

In [ ]:
# show rise_df

# 1. Setup: Get the list of words we are currently plotting
words_to_plot = rise_df['Word'].tolist()

# 2. Get the specific data slices
# A. Probabilities (You already have this)
probs_subset = df_final.loc[words_to_plot, ['Turn 1', 'Turn 2', 'Turn 3', 'Turn 4']]

# B. Absolute Counts (Retrieve from the df_counts we created earlier)
# We transpose (.T) so rows are Words and columns are Turns, matching probs_subset
counts_subset = df_counts[words_to_plot].T 

# C. Multiplication Factors (Current Prob / Turn 1 Prob)
# If a word drops from 0.10 to 0.05, Factor is 0.5x
factors_subset = probs_subset.div(probs_subset['Turn 1'], axis=0)

# 3. Melt all three individually
# We use reset_index() so 'Word' becomes a column
melt_probs = probs_subset.reset_index().melt(id_vars='index', var_name='Turn', value_name='Probability').rename(columns={'index': 'Word'})
melt_counts = counts_subset.reset_index().melt(id_vars='index', var_name='Turn', value_name='Count').rename(columns={'index': 'Word'})
melt_factor = factors_subset.reset_index().melt(id_vars='index', var_name='Turn', value_name='Factor').rename(columns={'index': 'Word'})

# 4. Merge them into one "Rich" DataFrame
df_rich = melt_probs.merge(melt_counts, on=['Word', 'Turn'])\
                    .merge(melt_factor, on=['Word', 'Turn'])

# Preview to ensure it looks right
print(df_rich.head())

In [ ]:
fig = px.line(
    df_rich, 
    x="Turn", 
    y="Probability", 
    color="Word", 
    markers=True,
    title="Evolution of User Language - rise words",
    template="plotly_white",
    height=500,
    # --- CUSTOM HOVER CONFIGURATION ---
    hover_data={
        "Turn": False,             # Hide (already on x-axis)
        "Word": False,             # Hide (already in legend)
        "Probability": ":.4f",     # Format: 4 decimal places
        "Count": True,             # Show raw count
        "Factor": ":.2f"           # Format: 2 decimal places
    }
)

# Optional: Rename the labels in the tooltip to be friendlier
fig.update_traces(
    hovertemplate="<br>".join([
        "<b>%{x}</b>",
        "Prob: %{y:.4f}",
        "Count: %{customdata[1]}",    # Accesses 'Count'
        "Growth: %{customdata[2]:.2f}x" # Accesses 'Factor'
    ])
)

fig.update_layout(yaxis_title="Probability (Frequency)", hovermode="x unified")
fig.show()